# RAG Document Question Answering

A compact walkthrough of a Retrieval Augmented Generation (RAG) pipeline.

```
PDF/TXT → Chunks → Embeddings → FAISS Vector Store → Question → Retrieval → LLM → Answer
```

Run the cells in order. The notebook works with a PDF or a plain text file.

If you do not set an API key, it falls back to a local answer helper.

## Step 0: Install dependencies

Run this once in the notebook environment.

In [5]:
%pip install -q langchain langchain-community langchain-text-splitters faiss-cpu sentence-transformers pypdf google-generativeai python-dotenv


Note: you may need to restart the kernel to use updated packages.


## Step 1: Imports

In [6]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

load_dotenv()

print("Imports successful ✅")


Imports successful ✅


## Step 2: Load the document

Set `DOCUMENT_PATH` to a PDF or TXT file.

In [7]:
DOCUMENT_PATH = r"pdf/VINIT_GAUTAM ML.pdf"  # replace with your file

file_extension = os.path.splitext(DOCUMENT_PATH)[1].lower()

if file_extension == '.pdf':
    loader = PyPDFLoader(DOCUMENT_PATH)
elif file_extension == '.txt':
    loader = TextLoader(DOCUMENT_PATH, encoding='utf-8')
else:
    raise ValueError('Unsupported file type. Please use a PDF or TXT file.')

documents = loader.load()

print(f"Loaded {len(documents)} document(s) from {DOCUMENT_PATH}")
print("\n--- Preview of page 1 ---\n")
print(documents[0].page_content[:500])

Loaded 2 document(s) from pdf/VINIT_GAUTAM ML.pdf

--- Preview of page 1 ---

VINIT GAUTAM
Jaipur, Rajasthan, India
+91 9511582356 — vinitgautam022@gmail.com
linkedin.com/in/vinit-gautam-ab41b11b4
github.com/vinitgautam022
leetcode.com/u/iamvinitgautam
PROFESSIONAL SUMMAR Y
Computer Science student with hands-on internship experience in full-stack development, machine learning, and data
engineering. Proficient inPython, JavaScript, and scalable system designwith a strong grasp of Object-Oriented
Programming (OOP) and scalable system design. Experienced in writing clean, m


## Step 3: Split the document

Split the text into overlapping chunks for retrieval.

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
print("\n--- Example chunk ---\n")
print(chunks[0].page_content)

Split into 5 chunks

--- Example chunk ---

VINIT GAUTAM
Jaipur, Rajasthan, India
+91 9511582356 — vinitgautam022@gmail.com
linkedin.com/in/vinit-gautam-ab41b11b4
github.com/vinitgautam022
leetcode.com/u/iamvinitgautam
PROFESSIONAL SUMMAR Y
Computer Science student with hands-on internship experience in full-stack development, machine learning, and data
engineering. Proficient inPython, JavaScript, and scalable system designwith a strong grasp of Object-Oriented
Programming (OOP) and scalable system design. Experienced in writing clean, maintainable code, collaborating on team
projects, and contributing to end-to-end application development across multiple technologies.
TECHNICAL SKILLS
Languages:Python, JavaScript, C++, SQL, HTML5, CSS3
F rameworks & T ools:Flask, REST APIs, Git, GitHub, VS Code, Jupyter Notebook
Concepts:OOP, Data Structures & Algorithms, ETL Pipelines, ML Model Development, Agile
Soft Skills:Analytical Thinking, Team Collaboration, Code Review Participation, Problem

## Step 4: Create embeddings

Each chunk becomes a vector using `all-MiniLM-L6-v2`.

The first run downloads the model.

In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Quick sanity check: embed a single sentence and see the vector shape
sample_vector = embeddings.embed_query("I love AI")
print(f"Vector length: {len(sample_vector)}")
print(f"First 5 values: {sample_vector[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector length: 384
First 5 values: [-0.04653948172926903, -0.058469515293836594, 0.04085371270775795, -0.04425811767578125, 0.06416134536266327]


## Step 5: Store the embeddings

FAISS keeps the chunk vectors searchable.

In [10]:
vectorstore = FAISS.from_documents(chunks, embeddings)

print("FAISS vector store created ✅")
print(f"Total vectors stored: {vectorstore.index.ntotal}")

FAISS vector store created ✅
Total vectors stored: 5


### Optional: save the index

Use this if you want to reuse the index later.

In [11]:
vectorstore.save_local("faiss_index")
print("Saved to ./faiss_index")

# To reload later instead of rebuilding, use:
# vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

Saved to ./faiss_index


## Step 6: Retrieve relevant chunks

The question is embedded the same way and matched against the stored vectors.

In [12]:
question = "explain my project"  # <-- change this to test different questions

relevant_chunks = vectorstore.similarity_search(question, k=3)

print(f"Retrieved {len(relevant_chunks)} relevant chunk(s):\n")
for i, chunk in enumerate(relevant_chunks, start=1):
    print(f"--- Chunk {i} (page {chunk.metadata.get('page', 'N/A')}) ---")
    print(chunk.page_content)
    print()

Retrieved 3 relevant chunk(s):

--- Chunk 1 (page 1) ---
Netflix Clone W ebsiteHTML5, CSS3, JavaScript
• Designed and implemented a pixel-accurate, fully responsive streaming platform clone with component-based frontend
structure and optimized layouts.
EDUCA TION
B.T ech in Computer Science Engineering2023 – Present
Poornima Institute of Engineering and Technology, Jaipur
Relevant Coursework: Data Structures & Algorithms, OOP, DBMS, Operating Systems, Software Engineering
ACHIEVEMENTS & CER TIFICA TIONS
•Finalist– Enigma 24-Hour Hackathon (competitive problem-solving under pressure)
•Solved150+ DSA problemson LeetCode, strengthening algorithmic and analytical thinking skills
•Certifiedin Data Engineering (Celebal Technologies), Data Structures & Algorithms and NPTEL Certifications
•Active open-source contributor on GitHub with multiple public repositories showcasing project work

--- Chunk 2 (page 0) ---
VINIT GAUTAM
Jaipur, Rajasthan, India
+91 9511582356 — vinitgautam022@gmail.com
li

## Step 7: Build the prompt

Combine the retrieved chunks with the question.

In [13]:
def build_prompt(relevant_chunks, question):
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    prompt = f"""You are a helpful assistant. Answer the question using ONLY
the context provided below. If the answer is not in the context,
say "I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:"""
    return prompt

prompt = build_prompt(relevant_chunks, question)
print(prompt)

You are a helpful assistant. Answer the question using ONLY
the context provided below. If the answer is not in the context,
say "I could not find this information in the document."

Context:
Netflix Clone W ebsiteHTML5, CSS3, JavaScript
• Designed and implemented a pixel-accurate, fully responsive streaming platform clone with component-based frontend
structure and optimized layouts.
EDUCA TION
B.T ech in Computer Science Engineering2023 – Present
Poornima Institute of Engineering and Technology, Jaipur
Relevant Coursework: Data Structures & Algorithms, OOP, DBMS, Operating Systems, Software Engineering
ACHIEVEMENTS & CER TIFICA TIONS
•Finalist– Enigma 24-Hour Hackathon (competitive problem-solving under pressure)
•Solved150+ DSA problemson LeetCode, strengthening algorithmic and analytical thinking skills
•Certifiedin Data Engineering (Celebal Technologies), Data Structures & Algorithms and NPTEL Certifications
•Active open-source contributor on GitHub with multiple public repositori

## Step 8: Answer generation

The notebook uses a local fallback unless a Gemini key is available.

If you want Gemini, add `GEMINI_API_KEY` to `.env`.

In [14]:
def answer_from_prompt(prompt):
    lines = [line.strip() for line in prompt.splitlines() if line.strip()]
    context_lines = []
    question_text = ""
    capture_context = False
    capture_question = False

    for line in lines:
        lower = line.lower()
        if lower == "context:":
            capture_context = True
            capture_question = False
            continue
        if lower == "question:":
            capture_context = False
            capture_question = True
            continue
        if lower == "answer:":
            capture_question = False
            continue
        if capture_context:
            context_lines.append(line)
        elif capture_question:
            question_text = line

    context_text = " ".join(context_lines).strip()
    if not context_text:
        return "I could not find this information in the document."

    import re

    sentences = re.split(r"(?<=[.!?])\s+", context_text)
    question_words = {
        word.lower()
        for word in re.findall(r"\b\w+\b", question_text)
        if len(word) > 2
    }

    scored_sentences = []
    for sentence in sentences:
        sentence_words = set(re.findall(r"\b\w+\b", sentence.lower()))
        score = len(question_words & sentence_words)
        scored_sentences.append((score, sentence.strip()))

    top_sentences = [sentence for score, sentence in sorted(scored_sentences, reverse=True) if sentence][:2]
    if not top_sentences:
        top_sentences = [context_text[:300]]

    if any(keyword in question_text.lower() for keyword in ["summary", "summarize", "summarise"]):
        return "Summary: " + " ".join(top_sentences)

    return " ".join(top_sentences)


## Step 9: Reusable helper

A small wrapper around retrieval and answer generation.

In [15]:
def ask(question, k=3, use_local=True, gemini_api_key=None):
    """
    Runs the full RAG pipeline for a single question against
    the already-built `vectorstore`.
    """
    relevant_chunks = vectorstore.similarity_search(question, k=k)
    prompt = build_prompt(relevant_chunks, question)

    api_key = gemini_api_key or os.getenv("GEMINI_API_KEY", "")

    if use_local or not api_key:
        answer = answer_from_prompt(prompt)
    else:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel("gemini-1.5-flash")
        answer = model.generate_content(prompt).text

    return answer, relevant_chunks


# Try it:
answer, chunks_used = ask("What is my educational background?", use_local=True)
print("🤖", answer)


🤖 •Collaborated with the backend team to integrate REST APIs and enhance application features based on user feedback. • Explored AI-based solutions to improve research outcomes, aligning with project goals through iterative experimentation.


## Step 10: Try your own questions

Edit the question and rerun the cell.

In [18]:
my_question = "my experince "
answer, chunks_used = ask(my_question, use_local=True)

print("❓ Question:", my_question)
print("\n🤖 Answer:", answer)
print("\n📚 Chunks used:")
for c in chunks_used:
    print("-", c.page_content[:100].replace("\n", " "), "...")

❓ Question: my experince 

🤖 Answer: •Participated in peer code reviews, contributing to best practices and maintainability of the codebase. •Collaborated with the backend team to integrate REST APIs and enhance application features based on user feedback.

📚 Chunks used:
- VINIT GAUTAM Jaipur, Rajasthan, India +91 9511582356 — vinitgautam022@gmail.com linkedin.com/in/vini ...
- cross-device compatibility. •Collaborated with the backend team to integrate REST APIs and enhance a ...
- Zeetron Networks • Developed and trainedML classification and regression modelsusing Python (scikit- ...


## Recap

1. Loaded the document
2. Split it into chunks
3. Created embeddings and stored them in FAISS
4. Retrieved the most relevant chunks for a query
5. Built a prompt and generated an answer



VINIT GAUTAM    
DATA SCIENCE INTERN 